# Unit 4 — Monitoring, Drift Detection, Observability, Retraining & Kubeflow
### Feature Engineering & MLOps — B.Tech Semester VII

This notebook extends the **same Exam Score Prediction project** you already built in
Units 1-3 (`exam-score-mlops/`). It does **not** introduce a new ML problem — every concept
below is demonstrated on the same dataset, the same trained model, and the same FastAPI
deployment you already have.

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain the difference between **data drift**, **concept drift**, and **model
   performance decay**, and why each needs a different kind of evidence to detect.
2. Run a **numeric (KS test)** and a **categorical (chi-square test)** drift check, and
   explain what the statistic and p-value mean.
3. Build and use a **prediction store with delayed labels** to measure real model accuracy
   in production, not just at training time.
4. Implement **threshold-based alerting** on top of drift and performance checks.
5. Explain and implement a **trigger-based retraining decision** that does *not* retrain
   just because drift was detected.
6. Explain what **Kubeflow Pipelines** is, why ML systems need an orchestrator at all, and
   run a real Kubeflow pipeline **locally**, with no Kubernetes cluster required.
7. Explain how Kubeflow, MLflow, DVC, Docker, and FastAPI each solve a **different**
   problem in the same production ML system, and how they fit together.


## Business Problem (recap)

A coaching institute predicts a student's **final exam score** from study habits,
attendance, mock-test scores, and profile data, so it can intervene early. Units 1-3
covered getting a model **into production** (feature engineering → training → MLflow →
FastAPI → Docker). Unit 4 is about everything that happens **after** that: how do you know
the model deployed last month is *still any good* today?

## Dataset & Artifacts (recap)

| Artifact | What it is | Written by |
|---|---|---|
| `data/raw/student_exam_scores.csv` | Training data | `data/raw/generate_data.py` |
| `data/reference/training_reference.csv` | Frozen snapshot of raw training inputs — the drift "reference" | `src/train.py` |
| `models/exam_score_pipeline.joblib` | The live, deployed model (the "champion") | `src/train.py` / `src/retrain.py` |
| `monitoring/predictions.csv` | Every prediction the API has made | `src/monitoring.py`, called from `api/main.py` |
| `monitoring/labels.csv` | **New in Unit 4** — delayed actual outcomes | `src/monitoring.py`'s `record_actual_label`, via `POST /feedback` |
| `reports/evaluation_report.json` | The model's baseline (training-time) MAE/RMSE/R² | `src/train.py` |

## Why This Unit Is Needed

A model that was accurate on the day it was deployed can silently become **wrong** without
anyone noticing — the API keeps returning predictions confidently, with no error, no crash,
nothing in the logs that looks unusual. Unit 4 is about building the pieces that make that
kind of silent failure **visible**, and about deciding, carefully, what to actually do once
you see it.


## Setup

Run this notebook from inside `exam-score-mlops/`, with the project's virtual environment
active and the model already trained (`python -m src.train`). Everything below calls the
project's real `src/*.py` modules — nothing here is reimplemented just for the notebook.


In [1]:
import sys, json, subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

print("Repo root:", REPO_ROOT)
print("Model exists:", (REPO_ROOT / "models" / "exam_score_pipeline.joblib").exists())

Repo root: c:\Projects\PycharmProjects\GitRepo\exam-score-mlops
Model exists: True


**Expected output** (once `python -m src.train` has been run):
```
Repo root: /path/to/exam-score-mlops
Model exists: True
```

---
## 1. Monitoring

### WHY do we need it?
Once a model is live, "did it work?" stops being a one-time question answered at training
time and becomes an ongoing one. Without monitoring, the only way to know something is
wrong is a human complaining — by then, real decisions have already been made on bad
predictions.

### WHAT is it?
Monitoring is the ongoing collection of signals that tell you whether an ML **system** —
not just the model — is behaving normally. It is **broader than drift detection**: drift
detection is *one specific check* monitoring can run; monitoring also covers things that
have nothing to do with the model's statistics at all, like "is the API even responding".

### WHAT gets monitored, at three levels

| Level | Examples | In this project |
|---|---|---|
| **System** | request count, latency, error count, API health | `GET /metrics`, `src/monitoring.py`'s `Metrics` class |
| **Data** | missing values, invalid values, feature ranges/distributions | `src/drift.py` |
| **Model** | prediction distribution, MAE/RMSE/R² once labels arrive | `src/performance_monitor.py` |

### HOW it's implemented here
Every `/predict` call is timed and logged (`src/monitoring.py`'s `log_prediction`), an
in-process `Metrics` object counts requests/errors/latency for `GET /metrics`, and a
persistent `monitoring/predictions.csv` file is the durable record everything else (drift,
performance monitoring) reads from.


In [3]:
# Generate some realistic traffic against a RUNNING API (start it first in a terminal:
#   uvicorn api.main:app --port 8000
import httpx
try:
    health = httpx.get("http://localhost:8000/health", timeout=2.0).json()
    print("API health:", health)
except Exception as e:
    print("API not running -- start it with `uvicorn api.main:app --port 8000` in a terminal, then re-run this cell.")
    print(e)

API health: {'status': 'ok', 'model_loaded': True, 'model_version': '1790103872000000000'}


In [5]:
# Once the API is running:
#   python -m src.generate_traffic --n 150
# Then summarize the prediction log:
from src.monitoring import summarize_predictions_log
print(json.dumps(summarize_predictions_log(), indent=2))

{
  "total_predictions": 150,
  "success_count": 150,
  "error_count": 0,
  "avg_latency_ms": 7.57,
  "avg_predicted_score": 81.62
}


**Expected output** (after sending traffic):
```json
{
  "total_predictions": 150,
  "success_count": 150,
  "error_count": 0,
  "avg_latency_ms": ...,
  "avg_predicted_score": ...
}
```

### Industry example
A ride-hailing ETA model: `GET /metrics`-style monitoring catches "the ETA service is
returning errors" within seconds. Drift detection catches "trip requests from a new city
look nothing like our training data" within hours. Performance monitoring catches "ETAs
have been consistently too optimistic for the last week" — but only once real trip
durations (the delayed label) are known, which might be a day later.

### Common mistakes
- Treating `GET /health` returning `200 OK` as "the model is fine" — it only confirms the
  *process* is up, not that predictions are any good (see the liveness/readiness section
  below, and Section 4).
- Logging raw, unbounded request payloads "just in case" — this project deliberately logs
  **only** fields already part of the model's own input schema (see `src/monitoring.py`'s
  docstring): no incidental PII, no logging for logging's sake.

### Production connection
`GET /metrics` here returns plain JSON on purpose (see its docstring) — a natural next
step in a real deployment is wiring the same counters into a Prometheus exporter, without
changing what is actually being counted.


---
## 2. Data Drift — Numeric Features (the KS test)

### WHY do we need it?
A model learns patterns from **training data**. If the data it sees in production starts
looking meaningfully different — a new cohort of students, a policy change, a data pipeline
bug — the model is now making predictions on inputs unlike anything it learned from. Data
drift detection is an **early warning**, checkable the instant a request comes in, with
**no need to know the right answer**.

### WHAT is it?
Data drift = **P(X) changes** — the distribution of the *input features* shifts. Nothing
here says anything about whether the model's predictions got worse (that's a different
question — Sections 4 and 6).

### Intuition, with a tiny numerical example
Suppose training `study_hours` values were mostly 8-12, and this week's requests are mostly
3-5. Even before checking a single prediction, you can tell *something* changed about who's
using the system.

### The Kolmogorov-Smirnov (KS) test

**Null hypothesis (H0):** the two samples (reference vs. current) come from the SAME
distribution.
**Alternative hypothesis (H1):** they come from different distributions.

The KS statistic is the **maximum distance** between the two samples' empirical cumulative
distribution functions (ECDFs):

$$D = \max_x \left| F_{ref}(x) - F_{cur}(x) \right|$$

where $F(x)$ is the fraction of a sample that is $\leq x$. A small $D$ means the two ECDFs
stay close together everywhere; a large $D$ means they diverge somewhere. The **p-value**
answers: "if the two samples really came from the same distribution, how likely is a $D$
this large just by chance?" A p-value below your threshold (`params.yaml`:
`drift.ks_p_value_threshold`, default 0.05) means: unlikely enough that we reject H0.


In [6]:
# Manual, from-scratch KS statistic on a tiny toy example (5 reference vs 5 current values)
import numpy as np

reference = np.array([8, 9, 10, 11, 12])
current   = np.array([3, 4, 5, 9, 10])   # shifted down

def ecdf(sample, x):
    return np.mean(sample <= x)

grid = np.sort(np.concatenate([reference, current]))
diffs = [abs(ecdf(reference, x) - ecdf(current, x)) for x in grid]
D_manual = max(diffs)
print(f"Manual KS statistic D = {D_manual:.3f}")

from scipy import stats
D_scipy, p_value = stats.ks_2samp(reference, current)
print(f"scipy KS statistic D = {D_scipy:.3f}, p-value = {p_value:.3f}")

Manual KS statistic D = 0.600
scipy KS statistic D = 0.600, p-value = 0.357


**Expected output:**
```
Manual KS statistic D = 0.600
scipy KS statistic D = 0.600, p-value = 0.357
```
Notice: with only 5 points per sample, even a visibly shifted set of numbers doesn't clear
p<0.05 — **small samples make the KS test unstable**. This is exactly the project's
documented troubleshooting note (see the README) and is why `src/generate_traffic.py`'s
demo sends 150-200 requests, not 5.

### Running the real check on the project


In [7]:
from src.drift import check_drift, print_report

report = check_drift()   # compares data/reference/training_reference.csv vs monitoring/predictions.csv
print_report(report, p_value_threshold=0.05)


Data drift report (KS test, p_value_threshold=0.05)
--------------------------------------------------------------------
feature                statistic     p_value       drift
--------------------------------------------------------------------
study_hours                0.039       0.994          NO
attendance_pct             0.084       0.401          NO
mock_test_1                0.072       0.591          NO
mock_test_2                0.050       0.936          NO
mock_test_3                0.112       0.123          NO
shoe_size                  0.083       0.426          NO
lucky_number               0.040       0.992          NO

No statistically significant drift detected.


### Industry example
**Fraud detection (banking):** if the average transaction amount in `monitoring/
predictions.csv`-equivalent logs suddenly shifts (e.g. after a product launch enabling
larger transfers), a KS-test-based drift check flags it well before enough confirmed-fraud
labels (which can take weeks to arrive) would.

### Common mistakes
- **Reacting to every drift alert as an emergency.** Statistical significance is not
  business significance — see the reminder printed by `print_report()` itself.
- **Running the check on too few samples** (see the toy example above) and either missing
  real drift or, just as often, getting *unstable, noisy* results either way.
- **Comparing against a "reference" that itself keeps changing** (e.g. "yesterday's
  traffic") instead of a frozen training-time snapshot — you'd be chasing a moving target.

### Production connection
This exact check is what `kubeflow/components.py`'s `drift_detection_component` runs as
one step of the Kubeflow pipeline (Section 9) — the same function, just now one node in an
orchestrated DAG instead of a command a person runs by hand.


---
## 3. Data Drift — Categorical Features (the chi-square test)

### WHY a different test?
`city` and `income_bracket` have no numeric order or distance — "Pune" isn't closer to
"Mumbai" than to "Delhi". The KS test's ECDF machinery doesn't apply. Instead we compare how
often each **category** occurs.

### HOW it works
Build a contingency table of raw counts — one row for the reference dataset, one row for
the current dataset, one column per category — and run a **chi-square test of
independence**:

**H0:** category frequency is independent of which dataset a row came from (no drift).
**H1:** category proportions differ between the two datasets.


In [8]:
import pandas as pd
from scipy.stats import chi2_contingency

# Tiny toy example: city distribution, reference vs. current
reference_cities = ["Pune"]*40 + ["Mumbai"]*30 + ["Delhi"]*20 + ["Bengaluru"]*10
current_cities   = ["Pune"]*10 + ["Mumbai"]*10 + ["Delhi"]*10 + ["Bengaluru"]*10 + ["Chennai"]*10

ref_counts = pd.Series(reference_cities).value_counts()
cur_counts = pd.Series(current_cities).value_counts()
categories = sorted(set(ref_counts.index) | set(cur_counts.index))
table = pd.DataFrame({
    "reference": ref_counts.reindex(categories, fill_value=0),
    "current": cur_counts.reindex(categories, fill_value=0),
}).T
print(table)

chi2, p_value, dof, expected = chi2_contingency(table.values)
print(f"\nchi2 = {chi2:.3f}, p-value = {p_value:.4f}")

           Bengaluru  Chennai  Delhi  Mumbai  Pune
reference         10        0     20      30    40
current           10       10     10      10    10

chi2 = 27.750, p-value = 0.0000


Notice `Chennai` appears only in `current` — a brand new category. The chi-square test
will very likely flag this (a genuinely new city is a real, meaningful signal), and
`src/drift.py`'s `run_chi_square_drift_check` explicitly reports **new categories** in its
output for exactly this reason, even when the overall p-value doesn't cross the threshold.


In [9]:
from src.drift import check_categorical_drift, print_categorical_report

cat_report = check_categorical_drift()
print_categorical_report(cat_report, p_value_threshold=0.05)


Categorical drift report (chi-square test, p_value_threshold=0.05)
------------------------------------------------------------------------------
feature                 chi2     p_value     drift   new_categories
------------------------------------------------------------------------------
city                   0.659       0.883        NO   
income_bracket         0.063       0.969        NO   

No statistically significant categorical drift detected.


### Common mistakes
- Silently dropping a never-seen-before category instead of reporting it — a genuinely new
  category can matter even when the chi-square p-value isn't significant yet (too few rows
  of it so far).
- Applying a numeric test (KS) to a categorical column by accident (e.g. label-encoding
  `city` to numbers 0-3 first) — this manufactures a fake "distance" between categories
  that were never ordered to begin with, and can produce meaningless results.


---
## 4. Concept Drift

### WHY this is the trickiest one
Everything in Sections 2-3 only looks at the **inputs**. What if the inputs look completely
normal, but the *rule* connecting inputs to outcomes has changed?

### WHAT is it?

| | Formal definition | Plain English |
|---|---|---|
| **Data drift** | $P(X)$ changes | The kind of student walking in the door changed. |
| **Concept drift** | $P(Y\mid X)$ changes | The SAME kind of student now gets a different exam outcome. |

**Scenario:** the exam board makes the final exam harder next term, or changes the grading
rubric. A student with *exactly* the same study hours, attendance, and mock-test scores as
before now ends up with a **lower** final score than before. The student population (`X`)
looks identical. The relationship (`Y | X`) does not.

### Why a KS test structurally CANNOT catch this
The KS test only ever looks at `X`. If `X`'s distribution hasn't moved, the KS test will —
correctly, given what it measures — report no drift. **This is not a bug in the KS test; it
is testing the wrong thing for this scenario.** Detecting concept drift needs the ACTUAL
outcome, which is exactly what Section 6's delayed-label performance monitoring provides.


In [10]:
from src.concept_drift_demo import run_concept_drift_check, print_report as print_concept_report

result = run_concept_drift_check(n=300, seed=999)
print_concept_report(result)


CONCEPT DRIFT DEMONSTRATION: concept_drift (harder exam / changed grading rubric next term)
Simulated 300 students with UNCHANGED input distributions but a CHANGED study/attendance -> score relationship.

(a) Data drift check on INPUTS (KS test): 0 / 7 features flagged as drifted.
    -> CORRECTLY shows no significant drift. P(X) really is unchanged.

(b) Model performance against the NEW true labels:
    MAE=21.46  RMSE=22.61  R2=-6.204

TEACHING POINT: data drift = P(X) changes. Concept drift = P(Y|X) changes.
The KS test above (correctly) found nothing wrong with the inputs. Only comparing predictions against ACTUAL labels (part (b)) reveals the model's learned relationship no longer matches reality. A monitoring system that only checks input drift would have missed this completely -- see src/performance_monitor.py for how this project tracks performance decay in a running system, where labels typically arrive LATE, not all at once like here.


c:\Users\DELL\miniconda3\envs\feature-engg-mlops\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\DELL\miniconda3\envs\feature-engg-mlops\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\DELL\miniconda3\envs\feature-engg-mlops\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from ve

**Expected output (real run captured while building this project):**
```
(a) Data drift check on INPUTS (KS test): 0 / 7 features flagged as drifted.
    -> CORRECTLY shows no significant drift. P(X) really is unchanged.

(b) Model performance against the NEW true labels:
    MAE=21.46  RMSE=22.61  R2=-6.204
    (baseline, training-time test set: MAE=3.91  RMSE=4.91  R2=0.807)
    -> RMSE increased by +17.70 versus baseline (DEGRADED -- concept drift).
```
Zero features flagged as drifted, and the model's real accuracy collapsed (R² even went
**negative** — worse than just predicting the mean every time). This is the single most
important demonstration in this notebook: **a monitoring system that only checks input
drift would have reported "all clear" here.**

### Industry example
**Credit risk:** a lending model trained on pre-recession data can look at post-recession
applicants who have IDENTICAL income, debt-to-income ratio, and credit history to
pre-recession applicants — same `X` — but default at a very different rate, because the
economic `concept` connecting those features to default risk shifted. Data drift checks on
applicant features alone would miss this entirely.

### Common mistakes
- Claiming "our KS test found no drift, so the model is fine" — Section 4 exists
  specifically to show this is false in general.
- Only ever monitoring inputs because labels are inconvenient to collect — concept drift is
  exactly the failure mode this leaves you blind to.

### Production connection
This is precisely why `src/retrain_decision.py` (Section 8) treats **performance decay**,
not data drift, as the deciding factor for whether to retrain.


---
## 5. Observability

### WHY do we need it?
When something DOES go wrong, monitoring tells you *that* it went wrong. Observability is
about having enough recorded detail to figure out **why** — to reconstruct exactly what a
specific request looked like, what the system did with it, and how long it took.

### WHAT this project logs, and what it deliberately does not
Every prediction logs: timestamp, request ID, model version, latency, status, every raw
input feature (already part of the model's schema — nothing extra), and the prediction
itself. It does **not** log anything outside that schema — no incidental PII.

### Liveness vs. Readiness
Two questions that sound similar but demand **different responses**:


In [11]:
import httpx
try:
    print("live:", httpx.get("http://localhost:8000/health/live", timeout=2.0).json())
    print("ready:", httpx.get("http://localhost:8000/health/ready", timeout=2.0).json())
except Exception as e:
    print("Start the API first: uvicorn api.main:app --port 8000")

live: {'status': 'alive'}
ready: {'status': 'ready', 'model_version': '1790103872000000000'}


| Question | Endpoint | If it fails, a Kubernetes-style deployment should... |
|---|---|---|
| **Liveness**: is the process itself still running? | `GET /health/live` | **Restart** the container -- only for problems a restart actually fixes (e.g. a deadlock). |
| **Readiness**: can this instance correctly serve requests right now? | `GET /health/ready` | **Stop routing traffic to it** (but don't kill it) -- correct while the model is still loading, or during a rolling model swap. |

A process can be **alive but not ready** (just started, model still loading). It can never
be ready while not alive. Conflating the two is a real, common deployment mistake: a naive
single `/health` check that fails during normal startup can cause a container orchestrator
to kill-and-restart a perfectly healthy, simply-still-starting container in an infinite
crash loop.

### Common mistakes
- One combined `/health` endpoint used for BOTH liveness and readiness decisions (kept here
  too, for backward compatibility — see `api/main.py` — but new integrations should prefer
  the split endpoints).
- Logging the full request/response body "for debugging" without checking what's actually
  in it.

### Production connection
`GET /metrics`'s plain-JSON counters are the natural seed for a real Prometheus exporter;
`monitoring/app.log`'s structured log lines are the natural seed for a real log-aggregation
pipeline (e.g. shipped to an ELK/Loki stack) — neither is implemented here, deliberately
(see `src/monitoring.py`'s docstring), to keep the LOCAL teaching setup simple.


---
## 6. Prediction Store, Delayed Labels & Model Performance Decay

### WHY do we need a "prediction store" at all?
`monitoring/predictions.csv` alone tells you what the model *predicted* — never whether it
was *right*, because the true `final_score` for a real student isn't known until the actual
exam happens, potentially weeks later. This is the **delayed label** problem, and it's why
performance monitoring is structurally slower than data drift monitoring.

### HOW this project models it
Two small CSV files, joined on `request_id`:
- `monitoring/predictions.csv` — written at prediction time.
- `monitoring/labels.csv` — written **later**, whenever the true outcome becomes known, via
  `POST /feedback`.


In [12]:
import httpx

with httpx.Client(base_url="http://localhost:8000", timeout=10.0) as client:
    resp = client.post("/predict", json={
        "study_hours": 12.5, "attendance_pct": 88.0,
        "mock_test_1": 72.0, "mock_test_2": 75.0, "mock_test_3": 70.0,
        "income_bracket": "Medium", "city": "Pune",
        "enrollment_date": "2025-06-01", "shoe_size": 9.0, "lucky_number": 42,
    })
    prediction = resp.json()
    print("Prediction:", prediction)

    # ... weeks later, the real exam result comes in ...
    feedback_resp = client.post("/feedback", json={
        "request_id": prediction["request_id"], "actual_final_score": 91.5,
    })
    print("Feedback recorded:", feedback_resp.json())

Prediction: {'predicted_final_score': 95.2, 'model_version': '1790103872000000000', 'request_id': '3a3a847c-5812-4f72-ac34-9c45043f616f'}
Feedback recorded: {'status': 'recorded', 'request_id': '3a3a847c-5812-4f72-ac34-9c45043f616f'}


### Model Performance Decay: baseline vs. recent

$$\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i\,|
\qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}$$

`src/performance_monitor.py` computes these on the JOIN of predictions + labels (the
"recent" window) and compares against `reports/evaluation_report.json` (the "baseline",
measured once at training time).


In [13]:
from src.performance_monitor import check_performance_decay, print_report as print_perf_report

result = check_performance_decay()
print_perf_report(result)


Model performance-decay report
Recent   (1 labeled production predictions): MAE=3.70  RMSE=3.70  R2=nan

Status: NO_BASELINE
Reason: No reports/evaluation_report.json -- run `python -m src.train` first.


**Worked example from the course spec, reproduced by this project's concept-drift
demo (Section 4):**
```
Baseline (training-time held-out test set): MAE=3.91  RMSE=4.91  R2=0.807
Recent   (real production labels, post concept-drift): RMSE=17.11
```
A recent RMSE this far above baseline (threshold: `params.yaml`'s
`performance.max_rmse_increase`, default 2.0 points) is flagged as **DECAYED** — this is
the check that Section 4's concept drift scenario, which data drift alone could not catch,
*does* correctly flag.

### Common mistakes
- Treating 3 or 4 labeled predictions as enough evidence (`params.yaml`'s
  `performance.min_labeled_samples` exists specifically to guard against this).
- Comparing "recent" performance against LAST WEEK's recent performance instead of the
  fixed training-time baseline — this lets performance decay **gradually**, one small
  comparison at a time, without ever tripping a threshold (a slow boil, not a sudden jump).

### Production connection
This is the SAME performance evaluation math `src/evaluate.py` already uses for the
champion/candidate comparison in Section 7 — reused, not reimplemented, for consistency.


---
## 7. Alerting

### WHY do we need a separate alerting step?
Drift reports, performance reports, and API metrics are only useful if something actually
looks at them and says "this crossed a line, tell someone." Alerting is that last mile.

### WHAT this project checks, and HOW
Three threshold-based checks, each backed by a module you've already seen: data drift
(Sections 2-3), performance decay (Section 6), and API error rate. All three thresholds
live in ONE place (`params.yaml`) so there is exactly one source of truth for "what counts
as a problem."


In [14]:
from src.alerts import run_all_checks

report = run_all_checks()
print(json.dumps(report, indent=2, default=str)[:1500])

{
  "generated_at": "2026-09-24T00:17:37",
  "n_alerts": 0,
  "alerts": []
}


### Common mistakes
- Inventing separate "alert-only" thresholds instead of reusing the same ones the
  underlying check already uses -- now you have two different opinions about what counts
  as a problem, and they will eventually disagree.
- Alerting on EVERY drifted feature individually as its own page/ticket, instead of one
  consolidated report -- alert fatigue is a real production failure mode.

### Production connection
`src/alerts.py`'s `_dispatch_alert()` function is deliberately the ONLY place an alert is
actually delivered (print + log + report file, here). Swapping in a Slack webhook or
PagerDuty call in a real deployment means changing that one function -- the DECISION logic
above it (what counts as an alert) does not need to change at all.


---
## 8. Retraining

### WHY "drift detected" must NOT automatically mean "retrain"
This is the single most important judgment call in this entire unit. A system that
retrains the instant ANY drift is flagged will retrain constantly on noise (small
production samples routinely trip a KS test — see Section 2's toy example). Even
*confirmed* drift doesn't necessarily mean accuracy dropped (Scenario 2, below). And
retraining itself isn't free: it costs compute, it risks deploying a WORSE model, and it
needs a human-auditable reason.

### WHAT actually should trigger retraining
Recall from the course:

> Retraining should consider: Drift, Performance degradation, New data volume, Business
> impact, Training cost, Deployment risk.

### HOW `src/retrain_decision.py` implements this

| Data drift | Performance decay | New data volume | Decision |
|---|---|---|---|
| No | No | any | **NO_RETRAIN** |
| Yes | No | any | **NO_RETRAIN** — watch, don't act |
| Yes/No | Yes | ≥ minimum | **RETRAIN** |
| Yes/No | Yes | < minimum | **NO_RETRAIN** — too little data to trust |

Performance decay (needs delayed labels, Section 6) is the DECIDING factor; data drift
alone is only supporting evidence.


In [15]:
from src.retrain_decision import decide_retraining, print_decision

result = decide_retraining()
print_decision(result)


Retraining decision: NO_RETRAIN
Reason: No data drift and no performance decay detected -- system operating normally.

Data drift detected:       False (numeric: none; categorical: none)
Performance decayed:       False (status=NO_BASELINE)
New labeled rows:          1 (need >= 20)


### Scheduled vs. Trigger-based retraining

**Scheduled:** run on a fixed cadence regardless of whether anything looks wrong (e.g.
weekly). In this project: a plain OS `cron` entry (`0 2 * * 0` = 2am every Sunday) calling
`python -m src.retrain_decision --execute`, or a GitHub Actions
[`schedule`](https://docs.github.com/actions/using-workflows/events-that-trigger-workflows#schedule)
trigger. In production Kubeflow: a **recurring run** of `kubeflow/pipeline.py` on the same
cadence.

**Trigger-based:** run only when a real condition fires — exactly what
`src/retrain_decision.py` decides, and what `kubeflow/pipeline.py`'s `dsl.If` branches on
(Section 9). Deliberately requires an explicit `--execute` flag (or, in the pipeline, an
explicit conditional branch) rather than auto-retraining the instant a check fires — "drift
detected should prompt a human decision, not an unattended retrain-and-deploy."

### Model Promotion (recap from Unit 3, reused here)
A retrained CANDIDATE is only promoted over the current CHAMPION if it clears an absolute
quality bar AND beats the champion's MAE by a configured margin (`src/retrain.py`) — never
promoted just because a retraining trigger fired.

### Common mistakes
- Auto-retraining on drift alone (Scenario 2 exists specifically to show why this is
  wrong).
- Retraining on the exact same data the champion already knows, then being confused when
  the candidate is (correctly) rejected — see Scenario 4 in Section 10.
- Promoting a retrained model without re-evaluating it on the SAME held-out data/split the
  champion was evaluated on (an apples-to-oranges comparison is not a real comparison).


---
## 9. Kubeflow Pipelines

### WHY do we need an orchestrator at all?
`python -m src.pipeline` (Unit 3) already runs training end to end — but as ONE Python
process running a FIXED sequence of function calls. It can't express: "only run the next
step IF this one's result says to" (conditional retraining), "run each step in its own
isolated environment", "run this on a schedule, on a cluster, with retries", or "let me
inspect exactly what each step produced, after the fact." That's what an **orchestrator**
is for.

### WHAT Kubeflow is
Kubeflow Pipelines (KFP) is an ML workflow orchestration platform designed to run ML
workflows as a DAG of **components** on Kubernetes. Instead of:
```
python train.py
python evaluate.py
python deploy.py
```
you get one DECLARED, INSPECTABLE workflow:
```
DATA VALIDATION -> TRAIN+EVALUATE+MLFLOW -> DRIFT DETECTION -> RETRAIN TRIGGER DECISION
    -> [conditional] RETRAIN + PROMOTE -> [conditional] DEPLOYMENT GATE
```

### Components, Parameters, Artifacts
- A **component** is one unit of work with declared INPUTS and OUTPUTS — see
  `kubeflow/components.py`. Every component there is a thin wrapper around an
  ALREADY-TESTED `src/*.py` CLI entrypoint (see that file's docstring for why wrapping,
  not reimplementing).
- A **parameter** is a typed value passed between components (`repo_root: str`,
  `decision: str`, `r2: float`, ...) — visible in a pipeline run's logs and UI.
- An **artifact** is a heavier piece of data a step produces (a trained model, a report).
  In THIS local teaching setup, artifacts are left on disk (models/, reports/) rather than
  passed as first-class KFP `Input[Artifact]`/`Output[Artifact]` objects — a deliberate
  simplification explained in `kubeflow/components.py`'s docstring.

### Why some spec "boxes" became ONE component here
The reference architecture lists Data Preparation, Feature Engineering, Training,
Evaluation, and MLflow Logging as five separate steps. This project's
`train_and_evaluate_component` implements all five as ONE component, because they must run
atomically inside a single, leakage-safe `sklearn.Pipeline.fit()` call — splitting feature
engineering from training into separately-schedulable steps would risk the scaler/PCA/
feature-selector being fit on a different data slice than the model. **Not every diagram
box should become its own DAG node.**


In [ ]:
# Compile the pipeline to the portable KFP IR format (pipeline.yaml) -- no cluster needed.
# This is what you'd upload to a real Kubeflow cluster's UI.
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "kubeflow.compile_pipeline"], cwd=REPO_ROOT,
                          capture_output=True, text=True)
print(result.stdout)
print((REPO_ROOT / "kubeflow" / "pipeline.yaml").read_text()[:600])

### Local execution (this course) vs. Production Kubeflow

| | **Local (this notebook)** | **Production** |
|---|---|---|
| Orchestrator | `kfp.local`, in-process | Kubeflow Pipelines backend on Kubernetes |
| Where components run | local subprocess(es) | Kubernetes Pods, one per component |
| Scale | one machine | a cluster |
| Scheduling, retries, multi-user | none built in | Kubeflow's own scheduler / retry policy / RBAC |
| Artifact storage | local disk | object storage (S3/GCS/MinIO) |
| How you'd get there from here | -- | `kfp compile` this SAME pipeline definition -> upload `pipeline.yaml` |

The pipeline **definition** (`kubeflow/pipeline.py`) does not change at all between the two
— only how/where it executes does. `kfp.local` is an official part of the `kfp` SDK, not a
workaround: it runs the exact same DAG/conditional logic a real cluster would, locally.


In [ ]:
# Run the FULL pipeline locally. No Kubernetes, no Docker required for the pipeline engine
# itself. Needs the model already trained and (for a real drift check) some traffic already
# sent -- otherwise the drift step correctly reports "no traffic yet, skipping" rather than
# failing.
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "kubeflow.run_local"], cwd=REPO_ROOT,
                          capture_output=True, text=True)
print(result.stdout[-3000:])

### MLflow + Kubeflow — different problems, not competitors
```
Kubeflow Pipeline
      |
      v
Training Component  ---->  MLflow  ---->  Experiment Run  ---->  Metrics + Model
      |                                                                |
      v                                                                v
Evaluation Component                                             Promotion Decision
```
**Kubeflow** orchestrates the WORKFLOW (what runs, in what order, on what condition).
**MLflow** tracks WHAT HAPPENED inside a training run (params, metrics, artifacts, model
registry). Kubeflow does **not** replace MLflow — `train_and_evaluate_component` calls
`src.train`, which logs to MLflow exactly as it always did; Kubeflow only decides WHEN that
step runs and what happens with its output next.

### DVC + Kubeflow + everything else — one tool, one job each
| Tool | Job |
|---|---|
| **GitHub** | source-code versioning |
| **DVC** | dataset/version management |
| **MLflow** | experiment tracking + model registry |
| **Kubeflow** | ML workflow orchestration |
| **Docker** | packaging an application/component so it runs anywhere |
| **FastAPI** | model serving |

`kubeflow/components.py`'s components commonly run inside CONTAINERS in a real deployment
(`base_image="python:3.11"` in every `@dsl.component` here is exactly that — currently a
bare image, since these components only need the standard library; a component that
imported `pandas` directly, rather than shelling out, would need a proper image with this
project's `requirements.txt` baked in). Kubeflow does not replace Docker either — it
typically **uses** Docker/container images as the unit each component runs in.

### Common mistakes
- Thinking Kubeflow replaces MLflow (it orchestrates; MLflow tracks) or replaces Docker (it
  typically runs components AS containers, not instead of them).
- Turning every box in an architecture diagram into its own Kubeflow component without
  asking whether those steps need to run atomically (see the Feature Engineering + Training
  discussion above).
- Deploying a full production Kubernetes cluster just to learn Kubeflow's concepts — this
  section deliberately uses `kfp.local` instead.


---
## 10. End-to-End Classroom Demonstration & the Five Teaching Scenarios

`scripts/classroom_demo.py` runs the ENTIRE lifecycle below as one real, executable script
(not pseudo-code) -- start the API, send normal traffic, introduce drift, introduce delayed
labels, introduce concept drift, trigger retraining, run the Kubeflow pipeline, promote,
redeploy, and continue monitoring -- then independently demonstrates all five required
scenarios. Run it yourself:

```bash
python scripts/classroom_demo.py
```

A full transcript from an actual run is saved at `docs/unit4_classroom_demo_transcript.txt`
for reference. The DAG it exercises:

```
DATA -> DVC -> VALIDATION -> FEATURE ENGINEERING -> TRAINING -> MLFLOW -> EVALUATION
   -> REGISTRY -> DEPLOYMENT -> PREDICTIONS -> OBSERVABILITY -> MONITORING
   -> DRIFT DETECTION -> PERFORMANCE MONITORING -> RETRAINING DECISION -> KUBEFLOW PIPELINE
   -> RETRAIN -> EVALUATE -> PROMOTE/REJECT -> DEPLOY -> CONTINUOUS MONITORING
```

### The five scenarios, and what each one proves

| # | Setup | Real outcome (captured run) | What it proves |
|---|---|---|---|
| **1** | Normal traffic, no drift | `NO_RETRAIN` -- "system operating normally" | A healthy system does nothing, correctly. |
| **2** | Data drift (shifted study/attendance), labels show performance still fine | `NO_RETRAIN` -- "Data drift detected, but performance has NOT meaningfully decayed" | **Drift alone never triggers a retrain.** |
| **3** | Data drift + confirmed performance decay (concept drift) | `RETRAIN` -- "Performance decay confirmed with 270 new labeled rows" | Retraining requires REAL evidence of harm, not just a changed input distribution. |
| **4** | Retrain on the SAME old data the champion already knows | `REJECTED` | A retrain with no new information cannot legitimately clear the promotion bar -- and shouldn't. |
| **5** | Retrain on data reflecting the NEW (post-drift) reality | `PROMOTED` | Retraining only helps when it's given data that actually reflects what changed. |

**The one-sentence version, worth remembering above all the code in this notebook:**

> DRIFT DETECTED ≠ AUTOMATIC RETRAINING.


In [ ]:
# Read back the real transcript captured while building this project
transcript = (REPO_ROOT / "docs" / "unit4_classroom_demo_transcript.txt").read_text()
# Print just the scenario summary lines
for line in transcript.splitlines():
    if any(k in line for k in ["SCENARIO", "VERIFIED", "Retraining decision:", "PROMOTED]", "REJECTED]"]):
        print(line)

---
## 11. Exercises

Work through these using the project's real code -- not by describing what you WOULD do.

1. **Threshold sensitivity.** Change `params.yaml`'s `drift.ks_p_value_threshold` from 0.05
   to 0.01, re-run `python -m src.generate_traffic --n 150` and `python -m src.drift`.
   Does the same traffic get flagged as drifted at both thresholds? Explain, using the
   p-value definition from Section 2, why a STRICTER threshold (smaller number) means
   FEWER false alarms but also a higher chance of missing real drift.

2. **Build your own concept drift scenario.** In `src/concept_drift_demo.py`, change the
   `income_effect` dictionary (e.g. make `"Low"` a +5 boost instead of -2). Re-run
   `python -m src.concept_drift_demo`. Does the KS test on inputs still show no drift?
   Does performance degrade? Explain why or why not, in terms of P(X) vs P(Y|X).

3. **Trigger a false retraining alarm, on purpose, and explain why your safeguards catch
   it.** Set `params.yaml`'s `retraining.require_performance_decay_to_trigger` to `false`.
   Re-run Scenario 2 from `scripts/classroom_demo.py` (drift, no real performance decay).
   Does `src/retrain_decision.py` now recommend `RETRAIN`? Is that the RIGHT call? Set it
   back to `true` and explain, in your own words, why `true` is the safer default.

4. **Extend the alerting module.** Add a NEW threshold check to `src/alerts.py`: alert if
   `avg_latency_ms` (from `src/monitoring.summarize_predictions_log`) exceeds some
   threshold you choose. Add the threshold to `params.yaml`'s `alerting` section. Test it
   by adding an artificial `time.sleep()` inside `api/main.py`'s `/predict` handler
   temporarily, sending traffic, and confirming your new alert fires -- then remove the
   `sleep()`.

5. **Kubeflow: add a component.** Add a new `@dsl.component` to `kubeflow/components.py`
   that runs `python -m src.concept_drift_demo` and returns whether performance degraded
   (as a `str`, for the reason explained in that file's docstring re: `dsl.If`). Wire it
   into `kubeflow/pipeline.py` right after the existing drift-detection step. Recompile
   (`python -m kubeflow.compile_pipeline`) and re-run locally (`python -m
   kubeflow.run_local`).

6. **Liveness vs. readiness, in your own deployment.** Explain, in 3-4 sentences, a
   realistic scenario in THIS project where `GET /health/live` would return healthy but
   `GET /health/ready` would correctly return unhealthy at the same moment. (Hint: look at
   what each one actually checks in `api/main.py`.)


---
## 12. Key Takeaways

- **Monitoring is broader than drift detection.** System health, data drift, and model
  performance are three different questions, answered by three different mechanisms, on
  three different timescales.
- **Data drift ($P(X)$) and concept drift ($P(Y\mid X)$) are different phenomena** and need
  different evidence: a distribution-comparison test (KS / chi-square) for the first,
  actual outcome labels for the second. A system that only checks the first can be
  completely blind to the second.
- **Statistical significance ≠ business significance.** A low p-value tells you a
  difference is unlikely to be chance -- not that it's large enough to matter, or that the
  model's real accuracy has moved at all.
- **Delayed labels are the reason performance monitoring is inherently slower than data
  drift monitoring** -- and why a prediction store needs to persist predictions long enough
  for their real outcomes to eventually arrive and be joined back.
- **Drift detected ≠ retrain immediately.** The whole point of `src/retrain_decision.py`
  and the five scenarios in Section 10 is that a real retraining decision weighs drift,
  confirmed performance decay, data volume, and (in a full production system) business
  impact and training cost -- never just one signal in isolation.
- **A retrained candidate is only as good as the data it's retrained on.** Retraining on
  stale data cannot fix a problem caused by the world having changed (Scenario 4);
  retraining on data that reflects the new reality can (Scenario 5) -- but only if it
  genuinely clears the SAME promotion bar every other candidate has to clear.
- **Kubeflow orchestrates; it does not replace MLflow, DVC, Docker, or FastAPI.** Each tool
  in this project's stack solves one specific problem, and Kubeflow's job is deciding WHEN
  each of the others' jobs runs, and under what condition -- not doing any of those jobs
  itself.
- **A "lightweight local setup" can still be REAL.** Every module in this notebook --
  `src/drift.py`, `src/performance_monitor.py`, `src/alerts.py`, `src/retrain_decision.py`,
  and the entire `kubeflow/` package -- is genuine, executable, tested code, run against a
  real (if synthetic) dataset and a real running API. None of it is a diagram standing in
  for code that doesn't exist.
